**Prueba**

In [1]:
import pandas as pd
import numpy as np

In [59]:
itc = pd.read_excel("base_app.xlsx")
cd  = pd.read_excel("base_clave_dinamica.xlsx")
mcd = pd.read_excel("maestro_demografico.xlsx")

Cada cliente solo tiene una fecha de vinculación al servicio de clave
dinámica.
En caso de que un cliente tenga más de una fecha de vinculación, se debe
tomar la fecha más reciente

In [62]:
cd = cd.sort_values(by='cfechacrea', ascending=False)
cd = cd.drop_duplicates(keep='first', subset='llave_id' )

Merges

In [65]:

mcd.columns

Index(['llave_cli', 'ctrl_terc', 'segm', 'f_vinc', 'ciudad_of', 'zona_of'], dtype='object')

In [66]:
cd.columns

Index(['llave_id', 'ccancrea', 'cusuario', 'cfechacrea', 'cod_of'], dtype='object')

In [67]:
itc.columns

Index(['documento', 'cdgtrn', 'cdgrpta', 'vlrtran', 'anotrn', 'mestrn',
       'diatrn'],
      dtype='object')

In [108]:
df = pd.merge(cd, mcd, left_on='llave_id', right_on='llave_cli', how='left').drop('llave_cli', axis=1)
df = pd.merge(df, itc, left_on='llave_id', right_on='documento', how='left')

- En un campo aparte, cuando el cdgrpta sea 0 se debe indicar que la
transacción es “Exitosa”, en caso contrario “No exitosa”.

In [109]:
df['ext_trasc'] = np.where(pr['cdgrpta'] == 0, 'Exitosa', 'No exitosa')

- Reemplazar los campos anotrn, mestrn y diatrn por el campo fechatrn en
formato numérico AAAAMMDD.

In [113]:
df['anotrn'] = df['anotrn'].astype(str)
df['mestrn'] = df['mestrn'].astype(str)
df['diatrn'] = df['diatrn'].astype(str)
df['fechatrn'] = df['anotrn'] + df['mestrn'] + df['diatrn']

- Adicionar un campo con el nombre del mes de vinculación al servicio de
clave dinámica
(Ejemplo: 2020/12/08 → Diciembre).

In [140]:
def mes(fecha):
  meses = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo',
    4: 'Abril', 5: 'Mayo', 6: 'Junio',
    7: 'Julio', 8: 'Agosto', 9: 'Septiembre',
    10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
  }
  return meses[int(fecha[4:6])]


In [141]:
df['mes_vinc'] = df['cfechacrea'].astype(str).apply(mes)

- Generar un campo con la cantidad de días entre la vinculación del cliente
al servicio de clave dinámica y la transacción.